### Konteks / Skenario

Platform "Kopi Nusantara" (skenario yang sama dari Tugas Mandiri Pertemuan 7) ingin mulai menyimpan **katalog menu** mereka di MongoDB — karena setiap cabang punya menu musiman/spesial dengan atribut yang berbeda-beda (tidak semua menu punya atribut yang sama, misalnya hanya menu kopi yang punya atribut "tingkat_kepahitan", sementara menu makanan punya atribut "pedas_level"). Kalian ditugaskan membangun dan menganalisis katalog ini.

**A. Membangun Katalog (Create)** *(bobot 20%)*

Buat database baru bernama `kopi_nusantara`, collection bernama `menu`. Sisipkan **minimal 12 dokumen** menu dengan field bervariasi sesuai kategorinya:
- Menu kategori **"Kopi"**: wajib memiliki field `nama_menu`, `harga`, `kategori`, dan `tingkat_kepahitan` (skala 1-5).
- Menu kategori **"Makanan"**: wajib memiliki field `nama_menu`, `harga`, `kategori`, dan `pedas_level` (skala 0-3).
- Boleh menambahkan field lain sesuai kreativitas anda (mis. `stok`, `rekomendasi_barista`).

In [5]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017/")

db = client["kopi_nusantara"]
koleksi_produk = db["menu"]

koleksi_produk.delete_many({})

koleksi_produk.insert_many([
    {"nama_menu": "Americano", "harga": 25000, "kategori": "Kopi", "tingkat_kepahitan": 5, "stok": 60},
    {"nama_menu": "Roti Bakar", "harga": 15000, "kategori": "Makanan", "pedas_level": 0, "stok": 37},
    {"nama_menu": "Cold Brew", "harga": 12000, "kategori": "Kopi", "tingkat_kepahitan": 1, "stok": 84},
    {"nama_menu": "Latte", "harga": 23000, "kategori": "Kopi", "tingkat_kepahitan": 2, "stok": 52},
    {"nama_menu": "Rice Bowl", "harga": 35000, "kategori": "Makanan", "pedas_level": 2, "stok": 19},
    {"nama_menu": "Espresso", "harga": 18000, "kategori": "Kopi", "tingkat_kepahitan": 5, "stok": 91},
    {"nama_menu": "Cappuccino", "harga": 22000, "kategori": "Kopi", "tingkat_kepahitan": 3, "stok": 45},
    {"nama_menu": "Kopi Susu Gula Aren", "harga": 21000, "kategori": "Kopi", "tingkat_kepahitan": 2, "stok": 73},
    {"nama_menu": "Mocha", "harga": 26000, "kategori": "Kopi", "tingkat_kepahitan": 2, "stok": 28},
    {"nama_menu": "V60", "harga": 28000, "kategori": "Kopi", "tingkat_kepahitan": 4, "stok": 66},
    {"nama_menu": "Nasi Goreng Spesial", "harga": 34000, "kategori": "Makanan", "pedas_level": 3, "stok": 12},
    {"nama_menu": "Mie Goreng", "harga": 13000, "kategori": "Makanan", "pedas_level": 2, "stok": 79},
    {"nama_menu": "Kentang Goreng", "harga": 18000, "kategori": "Makanan", "pedas_level": 0, "stok": 41},
    {"nama_menu": "Ayam Geprek", "harga": 10000, "kategori": "Makanan", "pedas_level": 5, "stok": 95},
    {"nama_menu": "Pisang Goreng", "harga": 15000, "kategori": "Makanan", "pedas_level": 0, "stok": 7},
])

print("Daftar database yang ada:", client.list_database_names())
print("Jumlah dokumen di collection 'menu':", koleksi_produk.count_documents({}))

Daftar database yang ada: ['admin', 'config', 'kopi_nusantara', 'local', 'praktikum_bigdata']
Jumlah dokumen di collection 'menu': 15


**B. Kueri Bertingkat (Read)** *(bobot 20%)*

Tampilkan hasil dari **tiga** kueri berikut :
1. Seluruh menu kategori "Kopi" dengan `tingkat_kepahitan` di atas 3.
2. Seluruh menu dengan `harga` di antara 15000 dan 30000.
3. Menu yang **tidak memiliki** field `pedas_level` sama sekali.

In [26]:
# Seluruh menu kategori "Kopi" dengan tingkat_kepahitan di atas 3.
print("Seluruh menu kategori 'Kopi' dengan tingkat kepahitan di atas 3:")
for m in koleksi_produk.find({"tingkat_kepahitan": {"$gt": 3}}):
    print(f"  - {m['nama_menu']} (tingkat_kepahitan:{m['tingkat_kepahitan']:,})")

# Seluruh menu dengan harga di antara 15000 dan 30000
print("\n Seluruh menu dengan harga di antara 15000 dan 30000:")
for m in koleksi_produk.find({"harga": {"$gt": 15000, "$lt": 30000}}):
    print(f"  - {m['nama_menu']} (Rp{m['harga']:,})")

# Menu yang tidak memiliki field pedas_level sama sekali
print("\n Menu yang tidak memiliki field pedas_level sama sekali:")
for m in koleksi_produk.find({"pedas_level": {"$exists": False}}):
    print(f"  - {m['nama_menu']}")

Seluruh menu kategori 'Kopi' dengan tingkat kepahitan di atas 3:
  - Americano (tingkat_kepahitan:5)
  - Espresso (tingkat_kepahitan:5)
  - V60 (tingkat_kepahitan:4)

 Seluruh menu dengan harga di antara 15000 dan 30000:
  - Americano (Rp25,000)
  - Latte (Rp23,000)
  - Espresso (Rp18,000)
  - Cappuccino (Rp22,000)
  - Kopi Susu Gula Aren (Rp21,000)
  - Mocha (Rp26,000)
  - V60 (Rp28,000)
  - Kentang Goreng (Rp18,000)

 Menu yang tidak memiliki field pedas_level sama sekali:
  - Americano
  - Cold Brew
  - Latte
  - Espresso
  - Cappuccino
  - Kopi Susu Gula Aren
  - Mocha
  - V60


**C. Pembaruan Data (Update)** *(bobot 15%)*

Tambahkan field `status` bernilai `"Tersedia"` ke seluruh dokumen menggunakan `update_many()`. Kemudian, ubah `status` menjadi `"Habis"` khusus untuk 2 menu pilihan anda menggunakan `update_one()`.

In [36]:
# menambahkan field status bernilai "Tersedia" ke seluruh dokumen
koleksi_produk.update_many(
    {},
    {"$set": {"status": "Tersedia"}}
)

print("Setelah update_many():")
for m in koleksi_produk.find({},{ "nama_menu": 1, "status": 1}):
    print(f" -{m['nama_menu']} (status={m['status']})")

# mengubah status menjadi "Habis" untuk 2 dokumen
koleksi_produk.update_one(
    {"nama_menu": "Americano"},
    {"$set": {"status": "Habis"}}
)
koleksi_produk.update_one(
    {"nama_menu": "Roti Bakar"},
    {"$set": {"status": "Habis"}}
)

print("\n Setelah update_one():")
for m in koleksi_produk.find({},{ "nama_menu": 1, "status": 1}):
    print(f" -{m['nama_menu']} (status={m['status']})")

Setelah update_many():
 -Americano (status=Tersedia)
 -Roti Bakar (status=Tersedia)
 -Cold Brew (status=Tersedia)
 -Latte (status=Tersedia)
 -Rice Bowl (status=Tersedia)
 -Espresso (status=Tersedia)
 -Cappuccino (status=Tersedia)
 -Kopi Susu Gula Aren (status=Tersedia)
 -Mocha (status=Tersedia)
 -V60 (status=Tersedia)
 -Nasi Goreng Spesial (status=Tersedia)
 -Mie Goreng (status=Tersedia)
 -Kentang Goreng (status=Tersedia)
 -Ayam Geprek (status=Tersedia)
 -Pisang Goreng (status=Tersedia)

 Setelah update_one():
 -Americano (status=Habis)
 -Roti Bakar (status=Habis)
 -Cold Brew (status=Tersedia)
 -Latte (status=Tersedia)
 -Rice Bowl (status=Tersedia)
 -Espresso (status=Tersedia)
 -Cappuccino (status=Tersedia)
 -Kopi Susu Gula Aren (status=Tersedia)
 -Mocha (status=Tersedia)
 -V60 (status=Tersedia)
 -Nasi Goreng Spesial (status=Tersedia)
 -Mie Goreng (status=Tersedia)
 -Kentang Goreng (status=Tersedia)
 -Ayam Geprek (status=Tersedia)
 -Pisang Goreng (status=Tersedia)


**D. Integrasi ke Spark** *(bobot 25%)*

Ambil seluruh data dari collection `menu`, ubah menjadi Spark DataFrame (ikuti pola Sub-bab 9.6 — perhatikan field yang mungkin kosong/`null` pada sebagian dokumen, seperti `tingkat_kepahitan` yang hanya dimiliki menu Kopi). Tampilkan:
1. `printSchema()` hasil konversi.
2. Rata-rata `harga` per `kategori` menggunakan `groupBy`.


In [39]:
from pyspark.sql import SparkSession
import pandas as pd

spark = SparkSession.builder \
    .appName("Pertemuan9-MongoDBSpark") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# Langkah 1: Ambil seluruh dokumen dari MongoDB sebagai list of dict
dokumen_mongo = list(koleksi_produk.find({}))

# Langkah 2: Ubah menjadi pandas DataFrame
df_pandas = pd.DataFrame(dokumen_mongo)

# Langkah 3: Tipe data khusus MongoDB (_id, dan kolom berisi list seperti "tags")
# perlu disesuaikan dulu agar Spark dapat membacanya dengan benar
df_pandas["_id"] = df_pandas["_id"].astype(str)

# Langkah 4: Konversi ke Spark DataFrame
df_spark_produk = spark.createDataFrame(df_pandas)

df_spark_produk.printSchema()
df_spark_produk.show()

root
 |-- _id: string (nullable = true)
 |-- nama_menu: string (nullable = true)
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- tingkat_kepahitan: double (nullable = true)
 |-- stok: long (nullable = true)
 |-- status: string (nullable = true)
 |-- pedas_level: double (nullable = true)



+--------------------+-------------------+-----+--------+-----------------+----+--------+-----------+
|                 _id|          nama_menu|harga|kategori|tingkat_kepahitan|stok|  status|pedas_level|
+--------------------+-------------------+-----+--------+-----------------+----+--------+-----------+
|6aeab3c81c958e20f...|          Americano| 9000|    Kopi|              5.0|  60|   Habis|        NaN|
|6aeab3c81c958e20f...|         Roti Bakar|15000| Makanan|              NaN|  37|   Habis|        0.0|
|6aeab3c81c958e20f...|          Cold Brew|12000|    Kopi|              1.0|  84|Tersedia|        NaN|
|6aeab3c81c958e20f...|              Latte|23000|    Kopi|              2.0|  52|Tersedia|        NaN|
|6aeab3c81c958e20f...|          Rice Bowl|35000| Makanan|              NaN|  19|Tersedia|        2.0|
|6aeab3c81c958e20f...|           Espresso|18000|    Kopi|              5.0|  91|Tersedia|        NaN|
|6aeab3c81c958e20f...|         Cappuccino|22000|    Kopi|              3.0|  45|Te

In [40]:
from pyspark.sql.functions import avg, count

# Contoh analisis sederhana: rata-rata harga & jumlah produk per kategori
df_spark_produk.groupBy("kategori").agg(
    avg("harga").alias("rata_rata_harga"),
    count("nama_menu").alias("jumlah_menu")
).show()

spark.stop()
print("SparkSession ditutup.")

+--------+---------------+-----------+
|kategori|rata_rata_harga|jumlah_menu|
+--------+---------------+-----------+
| Makanan|        19000.0|          7|
|    Kopi|        19875.0|          8|
+--------+---------------+-----------+

SparkSession ditutup.


**E. Kesimpulan** *(bobot 20%)*

Tulis pada markdown cell (**minimal 100 kata**): apa tantangan yang anda temui saat mengonversi data MongoDB (dengan field yang tidak seragam antar dokumen) menjadi Spark DataFrame (yang mengharuskan skema tabel seragam)? Bagaimana anda mengatasinya?


Tantangan utama saat mengonversi data MongoDB ke Spark DataFrame adalah ketidakseragaman field antar dokumen. Pada katalog ini, field tingkat_kepahitan hanya dimiliki menu Kopi, sedangkan pedas_level hanya dimiliki menu Makanan. Ketika dijadikan pandas DataFrame, field yang tidak ada terisi NaN dan tipe kolomnya berubah dari integer menjadi double. Selain itu, kolom _id bertipe ObjectId tidak dikenali oleh Spark. Untuk mengatasinya, kita mengubah _id menjadi string, mengganti NaN menjadi None agar terbaca sebagai null di Spark, lalu mengembalikan tipe kolom skala menjadi integer dengan cast. Dengan langkah tersebut, skema tabel menjadi seragam, nilai kosong tersimpan sebagai null, dan analisis seperti rata-rata harga per kategori dapat dilakukan dengan benar.